In [245]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import yfinance as yf
import ta
import math

# List Of Tradeable Pairs And Indicators

In [247]:
# Initialize MetaTrader 5 connection
mt5.initialize()

# Updated list of currency pairs
pairs = [
    "EURUSD",  # Euro / US Dollar
    "EURCHF",  # Euro / Swiss Franc
    "EURJPY",  # Euro / Japanese Yen
    "USDCHF",  # US Dollar / Swiss Franc
    "CHFJPY",  # Swiss Franc / Japanese Yen
    
    "USDJPY"   # US Dollar / Japanese Yen
]

currencies = [
   "DX-Y.NYB", # Dollar Currency Index
    "^XDE",    # Euro Currency Index
    "^XDS",    # Chf Currency Index
    "^XDN"     # Yen Currency Index
]

# Function to get the latest Ask and Bid prices for a given pair
def get_latest_prices(symbol):
    # Get the latest tick data for the symbol
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        print(f"Failed to get latest tick data for {symbol}")
        return None, None
    return tick.ask, tick.bid

# Function to get historical data for a given pair
def get_historical_data(symbol, timeframe=mt5.TIMEFRAME_M15, n_bars=64):
    # Fetch historical data
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, n_bars)
    if rates is None or len(rates) == 0:
        print(f"Failed to get historical data for {symbol}")
        return None
    data = pd.DataFrame(rates)
    data['time'] = pd.to_datetime(data['time'], unit='s')
    return data

# Function to calculate EMA, RSI, and ATR for a given pair
def calculate_indicators(symbol):
    # Get historical data for the pair
    data = get_historical_data(symbol)
    if data is None:
        return None, None, None, None

    # Calculate EMA 64
    data['EMA_64'] = ta.trend.ema_indicator(data['close'], window=64)

    # Calculate RSI 16
    data['RSI_16'] = ta.momentum.rsi(data['close'], window=16)

    # Calculate ATR 16
    data['ATR_16'] = ta.volatility.average_true_range(data['high'], data['low'], data['close'], window=16)

    # Get the latest values of the indicators
    latest_price = data['close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    latest_atr = data['ATR_16'].iloc[-1]

    return latest_price, latest_ema, latest_rsi, latest_atr

# Currency Index Data

In [249]:
def get_currency_data(currency):
    # Fetch DXY historical data (last 10 days, 15m interval)
    dxy = yf.Ticker(currency)
    data = dxy.history(period="10d", interval="15m")
    
    # Recalculate RSI & EMA
    data["RSI_16"] = ta.momentum.rsi(data["Close"], window=16)
    data["EMA_64"] = ta.trend.ema_indicator(data["Close"], window=64)

    latest_price = data['Close'].iloc[-1]
    latest_ema = data['EMA_64'].iloc[-1]
    latest_rsi = data['RSI_16'].iloc[-1]
    return latest_price, latest_ema, latest_rsi

def get_currencies_table(currencies):
    dict = {}
    for i in currencies:
        name = i
        if i == "DX-Y.NYB":
            name = "USD"
        elif i == "^XDE" :
            name = "EURO"
        elif i== "^XDS" :
            name = "CHF"
        elif i== "^XDN" :
            name = "YEN"
        
        latest_price, latest_ema, latest_rsi = get_currency_data(i)
        dict.update({name: [latest_price, latest_ema, latest_rsi ]})
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Price', 'EMA', 'RSI'])
    return data

get_currencies_table(currencies)

,Price,EMA,RSI
USD,106.876999,106.924451,41.560089
EURO,104.638000,104.577305,43.085344
CHF,110.889297,110.800036,39.863725
YEN,65.920403,65.584883,67.980244


# Pip Value

In [251]:
def get_pip_value(symbol):
    dec = 0.0001
    if "JPY" in symbol:
        dec = 0.01
    latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(symbol)
    pip_value = (dec * 100000) / latest_price
    return pip_value

# Position Size

In [253]:
def get_position_size(pair, stop_loss):
    balance = account_info.balance
    risk_amount = 0.005 * balance
    size = risk_amount / ( stop_loss * get_pip_value(pair))
    size = round(size, 2)
    return size

# Forex Pairs Data table

In [255]:
def get_data_table(pairs):
    dict = {}
    for i in pairs:
        ask, bid = get_latest_prices(i)
        latest_price, latest_ema, latest_rsi, latest_atr = calculate_indicators(i)
        dict.update({i: [ask,bid,latest_price, latest_ema, latest_rsi, latest_atr]})
    
    # Convert to DataFrame
    data = pd.DataFrame.from_dict(dict, orient='index', columns=['Ask', 'Bid', 'price', 'EMA', 'RSI', 'ATR'])
    
    data['Stop Loss']= data['ATR'] * 4
    data['Take Profit'] = data['Stop Loss'] * 2
    for i in pairs:
        dec = 10000
        if "JPY" in i:
            dec = 100
        stop_loss = round(data.loc[i, "Stop Loss"] * dec)
        take_profit = round(data.loc[i, "Take Profit"] * dec)
        data.loc[i,'Round Stop Loss'] = int(stop_loss)
        data.loc[i,'Round Take Profit'] = int(take_profit)
        data.loc[i,'Pip Value'] = get_pip_value(i)
        data.loc[i,'Position Size'] = get_position_size(pair, stop_loss)  
    return data
        
data = get_data_table(pairs)

data


,Ask,Bid,price,EMA,RSI,ATR,Stop Loss,Take Profit,Round Stop Loss,Round Take Profit,Pip Value,Position Size
EURUSD,1.04641,1.04637,1.04637,1.046097,56.069179,0.000626,0.002504,0.005009,25.0,50.0,9.556849,4.85
EURCHF,0.94367,0.94350,0.94350,0.943122,58.386832,0.000501,0.002006,0.004011,20.0,40.0,10.598834,6.07
EURJPY,158.73700,158.71500,158.71500,158.772148,51.424037,0.127400,0.509598,1.019196,51.0,102.0,6.300602,2.38
USDCHF,0.90181,0.90169,0.90169,0.901586,51.434681,0.000475,0.001901,0.003802,19.0,38.0,11.090286,6.39
CHFJPY,168.23800,168.19800,168.19800,168.322248,45.265753,0.115715,0.462862,0.925723,46.0,93.0,5.945374,2.64
USDJPY,151.68900,151.68400,151.68400,151.782129,46.052850,0.095087,0.380347,0.760694,38.0,76.0,6.592697,3.19


In [256]:
data.loc['USDJPY', 'Position Size']

np.float64(3.19)